# V6.1 FINAL — Merge + Cross-Source Audit + Freeze

Đây là notebook **nên chạy tiếp theo** trước khi train thêm bất kỳ model nào.

Nó làm 6 việc:

1. Restore đúng frozen base:
   `curated_v6_rs1280_56b7ea194ee5.tar`
2. Tự tải **FOTL_Drone** từ official GitHub và build overlay:
   - `nest -> bird_nest`
   - `kite -> foreign_object (subtype=kite)`
   - `balloon -> foreign_object (subtype=balloon)`
   - `person -> foreign_object (subtype=person)`
   - `fire`, `monkey` không merge
3. Audit:
   - class IDs / bbox;
   - split counts;
   - exact duplicates trong base;
   - exact duplicates giữa base ↔ FOTL;
   - optional perceptual near-duplicate scan base ↔ FOTL.
4. Tạo **runtime union** bằng symlink, không copy lại ~20GB base.
5. Xuất bảng cuối:
   - image count / box count từng class và split;
   - foreign-object subtype;
   - source registry;
   - provenance manifest.
6. Freeze V6.1:
   - overlay archive;
   - `freeze.json`;
   - manifests;
   - dataset report;
   - upload về Camber.

> Không train trong notebook này. Dataset phải được khóa trước, rồi YOLO/RT-DETR mới dùng cùng một fingerprint.

---

## Freeze principle

Frozen V6 base **không bị sửa**.

V6.1 được định nghĩa bằng:

```text
V6.1 =
    frozen base 56b7ea194ee5
  + frozen FOTL overlay <overlay_sha>
  + mapping policy
  + split policy
  + audit result
```

Nhờ vậy không cần tạo thêm một tar 20+ GB chỉ để nói rằng dataset đã đổi.


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, re, tarfile, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone

WORK = Path("/kaggle/working")
TMP = Path("/tmp/v6p1_final")
BASE_EXTRACT = TMP/"base_extract"
FOTL_RAW = TMP/"FOTL_Drone"
OVERLAY = WORK/"v6p1_fotl_overlay"
RUNTIME = TMP/"v6p1_runtime"
REPORTS = WORK/"v6p1_freeze_records"

for p in [TMP, BASE_EXTRACT, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Frozen inputs
# ---------------------------------------------------------------------
BASE_TAG = "56b7ea194ee5"
BASE_STASH = "stash://bunpmc/projects/yolov11sdi/datasets/curated_v6_rs1280_56b7ea194ee5.tar"

FOTL_REPO = "https://github.com/Changping-Li/FOTL_Drone.git"
FOTL_PAPER_DOI = "10.3390/drones8080361"

EXPECTED_NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

EXPECTED_BASE_SPLITS = {
    "train": 16910,
    "val": 2182,
    "test": 1820,
}

# ---------------------------------------------------------------------
# Policy
# ---------------------------------------------------------------------
SEED = 42
FOTL_VAL_FRACTION_FROM_SOURCE_TRAIN = 0.10

INCLUDE_FOTL_NEST = True
INCLUDE_FOTL_KITE = True
INCLUDE_FOTL_BALLOON = True
INCLUDE_FOTL_PERSON = True

# Exact duplicates are always removed from the *overlay*, never from frozen base.
DROP_EXACT_OVERLAY_DUPLICATES = True

# pHash scan is useful before publication/final benchmark.
# It can take several minutes because it decodes ~20k images.
RUN_NEAR_DUP_AUDIT = True
NEAR_DUP_HAMMING_THRESHOLD = 4

# If pHash finds candidates, notebook creates a review CSV/contact sheets.
# It does NOT automatically delete near duplicates because pHash can false-positive.
ALLOW_FREEZE_WITH_NEAR_DUP_CANDIDATES = True

UPLOAD_TO_CAMBER = True
CAMBER_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p1"


In [ ]:
# Install dependencies.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyyaml", "pandas", "pillow", "imagehash",
    "iterative-stratification"
])

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageOps, ImageDraw
import imagehash
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk=4*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def images_under(root):
    root = Path(root)
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )

print("Environment ready.")


## 1. Camber auth + restore exact frozen base


In [ ]:
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Thiếu CAMBER_API_KEY trong Kaggle Secrets.")
os.environ["CAMBER_API_KEY"] = key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:"
    f"{Path.home()}/.local/bin:"
    + os.environ.get("PATH","")
)
CAMBER = shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src, dst, required=True):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(src), str(dst)],
        text=True,
        capture_output=True,
        env=os.environ.copy()
    )
    ok = r.returncode == 0 and dst.exists()
    if required and not ok:
        raise RuntimeError(
            f"Camber cp failed:\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}"
        )
    return ok

def stash_put(src, remote, required=True):
    src = Path(src)
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(src), str(remote)],
        text=True,
        capture_output=True,
        env=os.environ.copy()
    )
    ok = r.returncode == 0
    if required and not ok:
        raise RuntimeError(
            f"Camber upload failed:\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}"
        )
    if ok:
        print("Uploaded:", remote)
    return ok

archive = TMP/"curated_v6_rs1280_56b7ea194ee5.tar"

if not any(BASE_EXTRACT.rglob("data.yaml")):
    print("Downloading exact frozen V6 base...")
    stash_cp(BASE_STASH, archive)
    print("Extracting...")
    subprocess.check_call(["tar", "-xf", str(archive), "-C", str(BASE_EXTRACT)])
    archive.unlink(missing_ok=True)

yaml_candidates = list(BASE_EXTRACT.rglob("data.yaml"))
if not yaml_candidates:
    raise RuntimeError("Không tìm thấy data.yaml sau khi extract base.")

print("data.yaml candidates:", yaml_candidates)


In [ ]:
# Detect the correct base dataset root by exact class gate.
def normalize_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x:int(x))]
    return list(names)

BASE = None
BASE_YAML = None

for yp in yaml_candidates:
    try:
        cfg = yaml.safe_load(yp.read_text())
        names = normalize_names(cfg.get("names", []))
        if names == EXPECTED_NAMES:
            BASE = yp.parent
            BASE_YAML = yp
            break
    except Exception:
        pass

if BASE is None:
    raise RuntimeError(
        "Không có data.yaml nào match đúng frozen 6-class taxonomy:\n"
        + "\n".join(map(str, yaml_candidates))
    )

print("✅ BASE:", BASE)
print("✅ BASE classes:", EXPECTED_NAMES)


## 2. Strict base audit

Nếu split count/class gate không đúng, notebook **dừng ngay**.  
Không có chuyện "chắc cũng là V6" nữa.


In [ ]:
base_split_counts = {}

for sp in ["train","val","test"]:
    p = BASE/"images"/sp
    if not p.exists():
        raise RuntimeError(f"Missing base split: {p}")
    n = len(images_under(p))
    base_split_counts[sp] = n

print("Base split counts:", base_split_counts)

assert base_split_counts == EXPECTED_BASE_SPLITS, (
    f"Wrong base split counts. Expected {EXPECTED_BASE_SPLITS}, "
    f"got {base_split_counts}"
)
print("✅ Exact frozen split gate passed.")


In [ ]:
# Validate every base YOLO label and compute authoritative class statistics.
def label_path_for_image(img, image_root, label_root):
    rel = img.relative_to(image_root)
    return label_root/rel.with_suffix(".txt")

base_rows = []
base_class_boxes = Counter()
base_class_images = Counter()
invalid_rows = []

for sp in ["train","val","test"]:
    iroot = BASE/"images"/sp
    lroot = BASE/"labels"/sp

    for ip in images_under(iroot):
        lp = label_path_for_image(ip, iroot, lroot)
        seen = set()
        nboxes = 0

        if lp.exists():
            for row_no, line in enumerate(lp.read_text(errors="ignore").splitlines(), 1):
                if not line.strip():
                    continue
                parts = line.split()
                try:
                    if len(parts) < 5:
                        raise ValueError("fewer than 5 fields")
                    cid = int(float(parts[0]))
                    vals = list(map(float, parts[1:5]))
                    if not (0 <= cid < len(EXPECTED_NAMES)):
                        raise ValueError(f"class id {cid}")
                    if not all(0.0 <= x <= 1.0 for x in vals):
                        raise ValueError(f"bbox {vals}")
                except Exception as e:
                    invalid_rows.append((str(lp), row_no, line, repr(e)))
                    continue

                base_class_boxes[(sp,cid)] += 1
                seen.add(cid)
                nboxes += 1

        for cid in seen:
            base_class_images[(sp,cid)] += 1

        base_rows.append({
            "split": sp,
            "source_group": "frozen_v6_base",
            "image_path": str(ip),
            "relative_image": ip.relative_to(iroot).as_posix(),
            "n_boxes": nboxes,
        })

if invalid_rows:
    bad = pd.DataFrame(
        invalid_rows,
        columns=["label","row","content","error"]
    )
    bad.to_csv(REPORTS/"base_invalid_labels.csv", index=False)
    raise RuntimeError(
        f"Base contains {len(invalid_rows)} invalid YOLO rows. "
        "See base_invalid_labels.csv"
    )

print("✅ All base label rows valid.")

for cid,name in enumerate(EXPECTED_NAMES):
    print("\n", cid, name)
    for sp in ["train","val","test"]:
        print(
            f"  {sp:5s} images={base_class_images[(sp,cid)]:5d} "
            f"boxes={base_class_boxes[(sp,cid)]:5d}"
        )


## 3. Base exact cross-split duplicate audit

Frozen base is not modified.  
If the exact same image bytes occur across different base splits, that is reported as a methodological issue.


In [ ]:
base_hash_rows = []
hash_to_splits = defaultdict(set)
hash_to_paths = defaultdict(list)

for row in base_rows:
    p = Path(row["image_path"])
    h = sha256_file(p)
    row["sha256"] = h
    base_hash_rows.append(row)
    hash_to_splits[h].add(row["split"])
    hash_to_paths[h].append(row)

base_cross_split_dups = {
    h: rows for h,rows in hash_to_paths.items()
    if len(hash_to_splits[h]) > 1
}

print("Base exact cross-split duplicate groups:", len(base_cross_split_dups))

if base_cross_split_dups:
    flat=[]
    for h,rows in base_cross_split_dups.items():
        for r in rows:
            flat.append({
                "sha256":h,
                "split":r["split"],
                "relative_image":r["relative_image"],
                "image_path":r["image_path"],
            })
    pd.DataFrame(flat).to_csv(
        REPORTS/"base_cross_split_exact_duplicates.csv",
        index=False
    )
    raise RuntimeError(
        "❌ Frozen base has exact cross-split duplicates. "
        "Do not freeze V6.1 until reviewed."
    )

print("✅ Base exact cross-split duplicate audit passed.")


## 4. Download FOTL_Drone and record exact git commit

This source directly strengthens `foreign_object`.

Important:
- repository class name is `ballon` in source config;
- we preserve it semantically as subtype `balloon`;
- source identity is never discarded.


In [ ]:
if not (FOTL_RAW/".git").exists():
    if FOTL_RAW.exists():
        shutil.rmtree(FOTL_RAW)
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        FOTL_REPO, str(FOTL_RAW)
    ])

FOTL_COMMIT = subprocess.check_output(
    ["git","-C",str(FOTL_RAW),"rev-parse","HEAD"],
    text=True
).strip()

print("✅ FOTL commit:", FOTL_COMMIT)

for sp in ["train","val"]:
    assert (FOTL_RAW/"images"/sp).exists()
    assert (FOTL_RAW/"labels"/sp).exists()

fotl_repo_counts = {
    sp: len(images_under(FOTL_RAW/"images"/sp))
    for sp in ["train","val"]
}
print("FOTL repo image counts:", fotl_repo_counts)


## 5. Build deterministic FOTL overlay

Source mapping:

| FOTL source | V6.1 |
|---|---|
| nest | bird_nest |
| kite | foreign_object / kite |
| ballon | foreign_object / balloon |
| fire | excluded |
| person | foreign_object / person |
| monkey | excluded |

Repo `train` is split 90/10 using multilabel stratification.  
Repo `val` is treated as source held-out set and mapped to V6.1 `test`.


In [ ]:
FOTL_NAMES = ["nest","kite","ballon","fire","person","monkey"]
TARGET_ID = {n:i for i,n in enumerate(EXPECTED_NAMES)}

FOTL_MAPPING = {
    0: ("bird_nest","nest") if INCLUDE_FOTL_NEST else (None,"nest"),
    1: ("foreign_object","kite") if INCLUDE_FOTL_KITE else (None,"kite"),
    2: ("foreign_object","balloon") if INCLUDE_FOTL_BALLOON else (None,"balloon"),
    3: (None,"fire"),
    4: ("foreign_object","person") if INCLUDE_FOTL_PERSON else (None,"person"),
    5: (None,"monkey"),
}

def parse_fotl(ip, source_split):
    iroot = FOTL_RAW/"images"/source_split
    lroot = FOTL_RAW/"labels"/source_split
    rel = ip.relative_to(iroot)
    lp = lroot/rel.with_suffix(".txt")

    kept=[]
    original=[]

    if lp.exists():
        for row_no,line in enumerate(lp.read_text(errors="ignore").splitlines(),1):
            if not line.strip():
                continue
            parts=line.split()
            if len(parts) < 5:
                raise RuntimeError(f"Malformed FOTL label: {lp}:{row_no}")

            cid=int(float(parts[0]))
            if cid not in FOTL_MAPPING:
                raise RuntimeError(f"Unknown FOTL class {cid}: {lp}")

            xywh=list(map(float,parts[1:5]))
            if not all(0.0 <= x <= 1.0 for x in xywh):
                raise RuntimeError(f"Invalid bbox: {lp}:{row_no} {xywh}")

            original.append(cid)
            target, subtype = FOTL_MAPPING[cid]
            if target is None:
                continue

            newparts=parts.copy()
            newparts[0]=str(TARGET_ID[target])

            kept.append({
                "source_row":row_no,
                "source_class_id":cid,
                "source_class":FOTL_NAMES[cid],
                "mapped_class_id":TARGET_ID[target],
                "mapped_class":target,
                "foreign_subtype":subtype if target=="foreign_object" else "",
                "yolo_line":" ".join(newparts),
            })

    return {
        "image":ip,
        "source_split":source_split,
        "source_relpath":rel.as_posix(),
        "original_classes":sorted(set(original)),
        "kept":kept,
    }

source_train = [
    parse_fotl(p,"train")
    for p in images_under(FOTL_RAW/"images"/"train")
]
source_held = [
    parse_fotl(p,"val")
    for p in images_under(FOTL_RAW/"images"/"val")
]

usable_train=[r for r in source_train if r["kept"]]
usable_test=[r for r in source_held if r["kept"]]

STRAT_LABELS=["bird_nest","kite","balloon","person"]

def strat_vec(r):
    present=set()
    for b in r["kept"]:
        if b["mapped_class"]=="bird_nest":
            present.add("bird_nest")
        elif b["mapped_class"]=="foreign_object":
            present.add(b["foreign_subtype"])
    return [int(x in present) for x in STRAT_LABELS]

X=np.arange(len(usable_train)).reshape(-1,1)
Y=np.asarray([strat_vec(r) for r in usable_train],dtype=np.int64)

msss=MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=FOTL_VAL_FRACTION_FROM_SOURCE_TRAIN,
    random_state=SEED,
)
idx_train, idx_val = next(msss.split(X,Y))

overlay_records=[]

for i in idx_train:
    r=usable_train[int(i)].copy()
    r["final_split"]="train"
    overlay_records.append(r)

for i in idx_val:
    r=usable_train[int(i)].copy()
    r["final_split"]="val"
    overlay_records.append(r)

for r0 in usable_test:
    r=r0.copy()
    r["final_split"]="test"
    overlay_records.append(r)

print("Overlay pre-dedup counts:", Counter(r["final_split"] for r in overlay_records))


## 6. Cross-source exact duplicate audit: Base ↔ FOTL

Frozen base always wins.

If an FOTL image is byte-identical to any image already in frozen V6:
- overlay copy is quarantined;
- base is untouched;
- the duplicate is recorded.


In [ ]:
base_hash_index = defaultdict(list)
for r in base_hash_rows:
    base_hash_index[r["sha256"]].append(r)

kept_overlay=[]
exact_dup_overlay=[]
overlay_internal_hashes=set()

for r in overlay_records:
    h=sha256_file(r["image"])
    r["sha256"]=h

    # Internal exact duplicate within FOTL overlay.
    if h in overlay_internal_hashes:
        exact_dup_overlay.append({
            "reason":"duplicate_inside_overlay",
            "overlay_split":r["final_split"],
            "overlay_file":r["source_relpath"],
            "sha256":h,
            "base_matches":"",
        })
        continue

    overlay_internal_hashes.add(h)

    # Exact duplicate against frozen base.
    if h in base_hash_index:
        exact_dup_overlay.append({
            "reason":"exact_duplicate_with_frozen_base",
            "overlay_split":r["final_split"],
            "overlay_file":r["source_relpath"],
            "sha256":h,
            "base_matches":" | ".join(
                f"{x['split']}:{x['relative_image']}"
                for x in base_hash_index[h]
            ),
        })
        if DROP_EXACT_OVERLAY_DUPLICATES:
            continue

    kept_overlay.append(r)

overlay_records=kept_overlay

dup_df=pd.DataFrame(exact_dup_overlay)
dup_path=REPORTS/"fotl_exact_duplicate_quarantine.csv"
dup_df.to_csv(dup_path,index=False)

print("Exact FOTL duplicates quarantined:", len(exact_dup_overlay))
print("Overlay post-exact-dedup counts:", Counter(r["final_split"] for r in overlay_records))


## 7. Optional perceptual near-duplicate audit

Exact SHA256 cannot catch:
- resize;
- recompression;
- small crop;
- JPEG quality change.

So this cell computes 64-bit pHash and finds FOTL ↔ frozen-base candidates.

`distance <= 4` is treated as **review candidate**, not automatic duplicate.

If candidates exist, contact sheets are generated in:

`/kaggle/working/v6p1_freeze_records/near_dup_contact_sheets/`


In [ ]:
near_dup_candidates=[]

def phash64(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        h = imagehash.phash(im, hash_size=8)
        return int(str(h), 16)

def hamming64(a,b):
    return (int(a)^int(b)).bit_count()

if RUN_NEAR_DUP_AUDIT:
    print("Computing base pHash...")
    base_ph=[]
    for i,r in enumerate(base_hash_rows,1):
        base_ph.append({
            "split":r["split"],
            "relative_image":r["relative_image"],
            "image_path":r["image_path"],
            "phash":phash64(r["image_path"]),
        })
        if i % 2000 == 0:
            print(" base", i, "/", len(base_hash_rows))

    base_hash_values=np.array([x["phash"] for x in base_ph],dtype=np.uint64)

    print("Scanning overlay against base...")
    for i,r in enumerate(overlay_records,1):
        q=phash64(r["image"])

        # A simple loop is reliable and only ~20k comparisons per overlay image.
        best_idx=-1
        best_dist=65
        for j,b in enumerate(base_ph):
            d=hamming64(q,b["phash"])
            if d < best_dist:
                best_dist=d
                best_idx=j
                if d == 0:
                    break

        if best_dist <= NEAR_DUP_HAMMING_THRESHOLD:
            b=base_ph[best_idx]
            near_dup_candidates.append({
                "overlay_split":r["final_split"],
                "overlay_file":r["source_relpath"],
                "overlay_path":str(r["image"]),
                "base_split":b["split"],
                "base_file":b["relative_image"],
                "base_path":b["image_path"],
                "phash_distance":best_dist,
            })

        if i % 100 == 0:
            print(" overlay", i, "/", len(overlay_records),
                  "candidates=", len(near_dup_candidates))

    near_df=pd.DataFrame(near_dup_candidates)
    near_path=REPORTS/"fotl_vs_base_near_duplicate_candidates.csv"
    near_df.to_csv(near_path,index=False)

    print("Near-duplicate candidates:", len(near_dup_candidates))

    # Contact sheets for human review.
    sheet_dir=REPORTS/"near_dup_contact_sheets"
    sheet_dir.mkdir(parents=True,exist_ok=True)

    def thumb(path,size=(360,260)):
        with Image.open(path) as im:
            im=ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail(size)
            canvas=Image.new("RGB",size,"white")
            x=(size[0]-im.width)//2
            y=(size[1]-im.height)//2
            canvas.paste(im,(x,y))
            return canvas

    for start in range(0,min(len(near_dup_candidates),80),8):
        chunk=near_dup_candidates[start:start+8]
        W,H=760,330*len(chunk)
        sheet=Image.new("RGB",(W,H),"white")
        draw=ImageDraw.Draw(sheet)

        for row_i,c in enumerate(chunk):
            y=row_i*330
            try:
                a=thumb(c["overlay_path"])
                b=thumb(c["base_path"])
                sheet.paste(a,(10,y+10))
                sheet.paste(b,(390,y+10))
            except Exception as e:
                draw.text((10,y+10),repr(e),fill="black")

            txt=(
                f"d={c['phash_distance']} | "
                f"FOTL {c['overlay_split']} ↔ BASE {c['base_split']}"
            )
            draw.text((10,y+280),txt,fill="black")

        sheet.save(sheet_dir/f"near_dup_{start:04d}.jpg",quality=90)

else:
    print("RUN_NEAR_DUP_AUDIT=False -> skipped.")


## 8. Build frozen FOTL overlay artifact

Only overlay images are archived.

The huge frozen base is referenced by its immutable base tag/path.


In [ ]:
if OVERLAY.exists():
    shutil.rmtree(OVERLAY)

for sp in ["train","val","test"]:
    (OVERLAY/"images"/sp).mkdir(parents=True,exist_ok=True)
    (OVERLAY/"labels"/sp).mkdir(parents=True,exist_ok=True)

def safe_name(rel):
    p=Path(rel)
    stem=re.sub(r"[^A-Za-z0-9_.-]+","_",p.stem)
    suffix=hashlib.sha1(rel.encode()).hexdigest()[:8]
    return f"fotl_{stem}_{suffix}{p.suffix.lower()}"

overlay_image_manifest=[]
overlay_box_manifest=[]

for r in overlay_records:
    sp=r["final_split"]
    out_name=safe_name(r["source_relpath"])
    out_img=OVERLAY/"images"/sp/out_name
    out_lbl=OVERLAY/"labels"/sp/Path(out_name).with_suffix(".txt")

    shutil.copy2(r["image"],out_img)
    out_lbl.write_text(
        "\n".join(b["yolo_line"] for b in r["kept"])+"\n",
        encoding="utf-8"
    )

    mapped=sorted({b["mapped_class"] for b in r["kept"]})
    subtypes=sorted({
        b["foreign_subtype"] for b in r["kept"]
        if b["foreign_subtype"]
    })

    overlay_image_manifest.append({
        "output_file":out_name,
        "final_split":sp,
        "source_dataset":"FOTL_Drone",
        "source_repo":FOTL_REPO,
        "source_commit":FOTL_COMMIT,
        "source_paper_doi":FOTL_PAPER_DOI,
        "source_repo_split":r["source_split"],
        "source_file":r["source_relpath"],
        "source_image_sha256":r["sha256"],
        "mapped_classes":",".join(mapped),
        "foreign_subtypes":",".join(subtypes),
        "n_boxes":len(r["kept"]),
        "license_note":"UNSPECIFIED in data repo; verify before redistribution",
    })

    for b in r["kept"]:
        overlay_box_manifest.append({
            "output_file":out_name,
            "final_split":sp,
            "source_dataset":"FOTL_Drone",
            "source_file":r["source_relpath"],
            **{k:v for k,v in b.items() if k!="yolo_line"},
        })

pd.DataFrame(overlay_image_manifest).to_csv(
    OVERLAY/"manifest_images.csv",index=False
)
pd.DataFrame(overlay_box_manifest).to_csv(
    OVERLAY/"manifest_boxes.csv",index=False
)

overlay_yaml={
    "path":str(OVERLAY),
    "train":"images/train",
    "val":"images/val",
    "test":"images/test",
    "names":EXPECTED_NAMES,
}
(OVERLAY/"data.yaml").write_text(
    yaml.safe_dump(overlay_yaml,sort_keys=False)
)

print("Overlay images:",len(overlay_image_manifest))
print("Overlay boxes:",len(overlay_box_manifest))


In [ ]:
# Compute deterministic overlay fingerprint.
h=hashlib.sha256()
h.update(BASE_TAG.encode())
h.update(FOTL_COMMIT.encode())
h.update(json.dumps(FOTL_MAPPING,sort_keys=True).encode())

for row in sorted(
    overlay_image_manifest,
    key=lambda x:(x["final_split"],x["output_file"])
):
    h.update(json.dumps(row,sort_keys=True,ensure_ascii=False).encode())
    lp=OVERLAY/"labels"/row["final_split"]/Path(row["output_file"]).with_suffix(".txt")
    h.update(lp.read_bytes())

OVERLAY_SHA=h.hexdigest()
OVERLAY_TAG=OVERLAY_SHA[:12]

print("OVERLAY_SHA =",OVERLAY_SHA)
print("OVERLAY_TAG =",OVERLAY_TAG)


## 9. Build runtime V6.1 union using symlinks

This is for validation and later training notebooks.

No image bytes from frozen base are copied.


In [ ]:
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)

for sp in ["train","val","test"]:
    for source in ["base","fotl"]:
        (RUNTIME/"images"/sp/source).mkdir(parents=True,exist_ok=True)
        (RUNTIME/"labels"/sp/source).mkdir(parents=True,exist_ok=True)

def symlink_tree_images_labels(src_root, split, source_name, out_root):
    iroot=src_root/"images"/split
    lroot=src_root/"labels"/split

    for ip in images_under(iroot):
        rel=ip.relative_to(iroot)
        lp=lroot/rel.with_suffix(".txt")

        dip=out_root/"images"/split/source_name/rel
        dlp=out_root/"labels"/split/source_name/rel.with_suffix(".txt")

        dip.parent.mkdir(parents=True,exist_ok=True)
        dlp.parent.mkdir(parents=True,exist_ok=True)

        os.symlink(ip,dip)

        if lp.exists():
            os.symlink(lp,dlp)
        else:
            dlp.write_text("")

for sp in ["train","val","test"]:
    symlink_tree_images_labels(BASE,sp,"base",RUNTIME)
    symlink_tree_images_labels(OVERLAY,sp,"fotl",RUNTIME)

runtime_yaml={
    "path":str(RUNTIME),
    "train":"images/train",
    "val":"images/val",
    "test":"images/test",
    "names":EXPECTED_NAMES,
}
(RUNTIME/"data.yaml").write_text(
    yaml.safe_dump(runtime_yaml,sort_keys=False)
)

print((RUNTIME/"data.yaml").read_text())


## 10. Authoritative V6.1 class distribution

This is the table to use from now on, not historical counts copied from old notebooks.


In [ ]:
final_class_boxes=Counter()
final_class_images=Counter()
final_split_images=Counter()
final_source_images=Counter()

for sp in ["train","val","test"]:
    iroot=RUNTIME/"images"/sp
    lroot=RUNTIME/"labels"/sp

    for ip in images_under(iroot):
        rel=ip.relative_to(iroot)
        lp=lroot/rel.with_suffix(".txt")

        source=rel.parts[0] if rel.parts else "unknown"
        final_split_images[sp]+=1
        final_source_images[(sp,source)]+=1

        seen=set()
        if lp.exists():
            for line in lp.read_text(errors="ignore").splitlines():
                if not line.strip(): continue
                parts=line.split()
                cid=int(float(parts[0]))
                if not (0 <= cid < len(EXPECTED_NAMES)):
                    raise RuntimeError(f"Bad final class id {cid}: {lp}")
                xywh=list(map(float,parts[1:5]))
                if not all(0.0 <= x <= 1.0 for x in xywh):
                    raise RuntimeError(f"Bad final bbox: {lp}")
                final_class_boxes[(sp,cid)]+=1
                seen.add(cid)

        for cid in seen:
            final_class_images[(sp,cid)]+=1

dist_rows=[]
for cid,name in enumerate(EXPECTED_NAMES):
    for sp in ["train","val","test"]:
        dist_rows.append({
            "class_id":cid,
            "class_name":name,
            "split":sp,
            "images_with_class":final_class_images[(sp,cid)],
            "boxes":final_class_boxes[(sp,cid)],
        })

dist_df=pd.DataFrame(dist_rows)
display(dist_df)

dist_df.to_csv(REPORTS/"v6p1_class_distribution.csv",index=False)

print("\nFinal split image counts:",dict(final_split_images))
print("Source image counts:")
for k,v in sorted(final_source_images.items()):
    print(k,v)


In [ ]:
# Foreign-object subtype report from overlay.
subtype_rows=[]

for subtype in ["kite","balloon","person"]:
    for sp in ["train","val","test"]:
        boxes=sum(
            1 for x in overlay_box_manifest
            if x["final_split"]==sp
            and x["mapped_class"]=="foreign_object"
            and x["foreign_subtype"]==subtype
        )
        images=len({
            x["output_file"] for x in overlay_box_manifest
            if x["final_split"]==sp
            and x["mapped_class"]=="foreign_object"
            and x["foreign_subtype"]==subtype
        })
        subtype_rows.append({
            "subtype":subtype,
            "split":sp,
            "images":images,
            "boxes":boxes,
        })

subtype_df=pd.DataFrame(subtype_rows)
display(subtype_df)
subtype_df.to_csv(REPORTS/"foreign_object_subtypes.csv",index=False)


## 11. Source registry

This registry separates:
- sources actually in frozen V6;
- FOTL overlay newly added;
- established reference datasets used as literature evidence;
- unresolved legacy provenance.

VPMBGI's dataset-paper DOI is deliberately **not asserted** here because the previously linked DOI metadata was inconsistent. Reuse evidence is stored separately.


In [ ]:
# Try to read any source metadata embedded in frozen artifact.
embedded_meta=[]
for p in BASE.rglob("*.json"):
    if "meta" in p.name.lower() or "manifest" in p.name.lower():
        try:
            embedded_meta.append({
                "path":str(p.relative_to(BASE)),
                "json":json.loads(p.read_text())
            })
        except Exception:
            pass

source_registry=[
    {
        "source_id":"V5_CURATED",
        "status":"IN_FROZEN_V6",
        "role":"legacy curated defect source",
        "official_url":"",
        "paper_doi":"",
        "license":"mixed/inherited; verify upstream source-by-source",
        "notes":"Do not re-merge; already in frozen base.",
    },
    {
        "source_id":"VPMBGI",
        "status":"IN_FROZEN_V6",
        "role":"broken-glass insulator",
        "official_url":"https://github.com/phd-benel/VPMBGI",
        "paper_doi":"UNRESOLVED",
        "license":"CC BY 4.0 for images per official repository",
        "reuse_evidence":"YOLOv8 broken-glass-insulator work; LMACNet work using public VPMBGI",
        "notes":"Previous DOI mapping was inconsistent; do not cite it as dataset DOI until verified.",
    },
    {
        "source_id":"PEID_YW",
        "status":"IN_FROZEN_V6",
        "role":"legacy foreign_object",
        "official_url":"",
        "paper_doi":"",
        "license":"UNRESOLVED",
        "notes":"Keep through frozen V6; resolve canonical provenance before direct citation.",
    },
    {
        "source_id":"CXS_BIRDNEST",
        "status":"IN_FROZEN_V6",
        "role":"bird_nest",
        "official_url":"",
        "paper_doi":"",
        "license":"UNRESOLVED",
        "notes":"Keep through frozen V6; resolve canonical provenance before direct citation.",
    },
    {
        "source_id":"PTLAI",
        "status":"IN_FROZEN_V6",
        "role":"power-line assets / selected fault states",
        "official_url":"https://github.com/freds0/PTL-AI_Furnas_Dataset",
        "paper_doi":"10.1109/SIBGRAPI55357.2022.9991806",
        "license":"verify dataset redistribution terms from official source",
        "notes":"Do not re-merge; already in frozen base.",
    },
    {
        "source_id":"FOTL_DRONE",
        "status":"ADDED_IN_V6P1_OVERLAY",
        "role":"foreign_object expansion + bird_nest",
        "official_url":"https://github.com/Changping-Li/FOTL_Drone",
        "paper_doi":FOTL_PAPER_DOI,
        "license":"UNSPECIFIED in data repo; verify before redistribution",
        "git_commit":FOTL_COMMIT,
        "mapping":"nest->bird_nest; kite/balloon/person->foreign_object; fire/monkey excluded",
        "notes":"Original subtype preserved in manifests.",
    },
    {
        "source_id":"INSPLAD",
        "status":"REFERENCE_NOT_MERGED",
        "role":"established power-line UAV benchmark / literature evidence",
        "official_url":"https://github.com/andreluizbvs/InsPLAD",
        "paper_doi":"10.1080/01431161.2023.2283900",
        "license":"CC BY-NC 3.0 on Mendeley dataset",
        "notes":"Detection boxes are assets; fault data is not directly defect-region compatible.",
    },
    {
        "source_id":"TTPLA",
        "status":"REFERENCE_NOT_MERGED",
        "role":"established transmission tower/power-line benchmark",
        "official_url":"https://github.com/R3ab/ttpla_dataset",
        "paper_doi":"10.1007/978-3-030-69544-6_36",
        "license":"verify official dataset terms",
        "notes":"Structure benchmark, not defect-region labels.",
    },
    {
        "source_id":"EFOD_DRONE",
        "status":"AUGMENTATION_ONLY_NOT_MERGED",
        "role":"optional train-only augmented derivative of FOTL",
        "official_url":"https://github.com/yyh7979/EFOD_Drone",
        "paper_doi":"",
        "license":"VERIFY",
        "notes":"Never count augmented siblings as independent val/test evidence.",
    },
]

pd.DataFrame(source_registry).to_csv(
    REPORTS/"source_registry.csv",index=False
)
(REPORTS/"source_registry.json").write_text(
    json.dumps(source_registry,indent=2,ensure_ascii=False)
)

(REPORTS/"embedded_base_metadata.json").write_text(
    json.dumps(embedded_meta,indent=2,ensure_ascii=False)
)

display(pd.DataFrame(source_registry))


## 12. Freeze readiness + final fingerprint

The final fingerprint incorporates:
- frozen base tag;
- exact FOTL commit;
- overlay fingerprint;
- taxonomy;
- mapping/split policy;
- exact-duplicate audit result.

Near-duplicate candidates are recorded separately because pHash is not a proof of duplication.


In [ ]:
near_dup_n=len(near_dup_candidates)

freeze_ready = (
    len(base_cross_split_dups)==0
    and (
        ALLOW_FREEZE_WITH_NEAR_DUP_CANDIDATES
        or near_dup_n==0
    )
)

freeze_core={
    "schema_version":"v6p1-freeze-1",
    "created_utc":utc_now(),
    "dataset_name":"V6.1",
    "base":{
        "tag":BASE_TAG,
        "stash":BASE_STASH,
        "split_counts":base_split_counts,
    },
    "overlay":{
        "source":"FOTL_Drone",
        "git_commit":FOTL_COMMIT,
        "sha256":OVERLAY_SHA,
        "tag":OVERLAY_TAG,
        "split_counts":dict(Counter(
            x["final_split"] for x in overlay_image_manifest
        )),
    },
    "taxonomy":EXPECTED_NAMES,
    "mapping_policy":{
        "nest":"bird_nest",
        "kite":"foreign_object",
        "balloon":"foreign_object",
        "person":"foreign_object",
        "fire":"excluded",
        "monkey":"excluded",
    },
    "split_policy":{
        "frozen_base":"unchanged",
        "FOTL_repo_train":"multilabel stratified 90/10 -> train/val",
        "FOTL_repo_val":"held-out -> test",
        "seed":SEED,
    },
    "audits":{
        "base_exact_cross_split_duplicate_groups":len(base_cross_split_dups),
        "overlay_exact_duplicates_quarantined":len(exact_dup_overlay),
        "near_duplicate_scan_enabled":RUN_NEAR_DUP_AUDIT,
        "near_duplicate_threshold":NEAR_DUP_HAMMING_THRESHOLD if RUN_NEAR_DUP_AUDIT else None,
        "near_duplicate_candidates":near_dup_n,
        "near_duplicate_policy":"manual review; not auto-dropped",
    },
    "final_split_counts":dict(final_split_images),
    "freeze_ready":freeze_ready,
}

fingerprint_material=json.dumps(
    freeze_core,
    sort_keys=True,
    ensure_ascii=False
).encode()

FINAL_SHA=hashlib.sha256(fingerprint_material).hexdigest()
FINAL_TAG=f"v6p1_{FINAL_SHA[:12]}"

freeze_core["final_sha256"]=FINAL_SHA
freeze_core["final_tag"]=FINAL_TAG

freeze_json=REPORTS/"v6p1.freeze.json"
freeze_json.write_text(
    json.dumps(freeze_core,indent=2,ensure_ascii=False)
)

print(json.dumps(freeze_core,indent=2,ensure_ascii=False))


## 13. Human-readable final dataset report


In [ ]:
lines=[]
lines.append("# V6.1 Frozen Dataset Report")
lines.append("")
lines.append(f"- Final tag: `{FINAL_TAG}`")
lines.append(f"- Final SHA256: `{FINAL_SHA}`")
lines.append(f"- Frozen base: `{BASE_TAG}`")
lines.append(f"- FOTL commit: `{FOTL_COMMIT}`")
lines.append(f"- FOTL overlay SHA256: `{OVERLAY_SHA}`")
lines.append(f"- Freeze ready: **{freeze_ready}**")
lines.append("")
lines.append("## Taxonomy")
for i,n in enumerate(EXPECTED_NAMES):
    lines.append(f"{i}. `{n}`")
lines.append("")
lines.append("## Final split image counts")
for sp in ["train","val","test"]:
    lines.append(f"- {sp}: {final_split_images[sp]}")
lines.append("")
lines.append("## Per-class distribution")
lines.append("")
lines.append(dist_df.to_markdown(index=False))
lines.append("")
lines.append("## Foreign-object subtype contribution from FOTL")
lines.append("")
lines.append(subtype_df.to_markdown(index=False))
lines.append("")
lines.append("## Audit")
lines.append(f"- Base exact cross-split duplicate groups: {len(base_cross_split_dups)}")
lines.append(f"- FOTL exact duplicates quarantined: {len(exact_dup_overlay)}")
lines.append(f"- Near-duplicate candidates (pHash): {near_dup_n}")
lines.append("")
lines.append("## Source policy")
lines.append("- Frozen base sources remain immutable.")
lines.append("- FOTL_Drone is the only new source merged in this V6.1 freeze.")
lines.append("- InsPLAD and TTPLA remain reference/evidence datasets, not silently merged.")
lines.append("- EFOD_Drone is augmentation-only and is not independent held-out evidence.")
lines.append("- VPMBGI dataset DOI remains unresolved in this registry until separately verified.")
lines.append("")
lines.append("## Research rule")
lines.append(
    "All subsequent YOLO / RT-DETR baseline and ablation runs must record "
    f"`dataset_tag={FINAL_TAG}`."
)

report_md=REPORTS/"V6P1_DATASET_REPORT.md"
report_md.write_text("\n".join(lines),encoding="utf-8")

print(report_md.read_text()[:6000])


## 14. Package overlay + freeze records

The base V6 is already frozen in Camber, so only the new overlay and audit records are archived.


In [ ]:
overlay_archive=WORK/f"v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"
if overlay_archive.exists():
    overlay_archive.unlink()

subprocess.check_call([
    "tar","-czf",str(overlay_archive),
    "-C",str(OVERLAY.parent),
    OVERLAY.name
])

records_zip=WORK/f"{FINAL_TAG}_freeze_records.zip"
if records_zip.exists():
    records_zip.unlink()

with zipfile.ZipFile(records_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in REPORTS.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(REPORTS.parent))

print("Overlay archive:",overlay_archive)
print("Overlay archive GB:",round(overlay_archive.stat().st_size/1024**3,3))
print("Freeze records:",records_zip)


## 15. Upload frozen V6.1 records to Camber

Remote layout:

```text
stash://bunpmc/projects/yolov11sdi/datasets/v6p1/<FINAL_TAG>/
    v6p1.freeze.json
    V6P1_DATASET_REPORT.md
    source_registry.csv
    source_registry.json
    v6p1_class_distribution.csv
    foreign_object_subtypes.csv
    fotl_exact_duplicate_quarantine.csv
    fotl_vs_base_near_duplicate_candidates.csv
    v6p1_fotl_overlay_<sha>.tar.gz
    <FINAL_TAG>_freeze_records.zip
```


In [ ]:
if not freeze_ready:
    raise RuntimeError(
        "❌ freeze_ready=False. Review audit outputs before uploading final dataset records."
    )

if UPLOAD_TO_CAMBER:
    remote_root=f"{CAMBER_ROOT}/{FINAL_TAG}"

    upload_files=[
        freeze_json,
        report_md,
        REPORTS/"source_registry.csv",
        REPORTS/"source_registry.json",
        REPORTS/"v6p1_class_distribution.csv",
        REPORTS/"foreign_object_subtypes.csv",
        REPORTS/"fotl_exact_duplicate_quarantine.csv",
        REPORTS/"embedded_base_metadata.json",
        overlay_archive,
        records_zip,
    ]

    near_path=REPORTS/"fotl_vs_base_near_duplicate_candidates.csv"
    if near_path.exists():
        upload_files.append(near_path)

    for p in upload_files:
        if p.exists():
            stash_put(p,f"{remote_root}/{p.name}")

    print("\n✅ FINAL DATASET TAG:",FINAL_TAG)
    print("✅ CAMBER ROOT:",remote_root)
else:
    print("UPLOAD_TO_CAMBER=False")
    print("FINAL DATASET TAG:",FINAL_TAG)


# What to do after this notebook

Do **not** change the dataset silently after this.

If the notebook ends with:

```text
✅ FINAL DATASET TAG: v6p1_xxxxxxxxxxxx
```

then that exact tag becomes the required dataset ID for:

- YOLO11n edge baseline;
- YOLO11m cloud baseline;
- RT-DETR-R50 B0;
- RT-DETR-R50 B1 Frequency;
- RT-DETR-R50 B2 Structural;
- StructDETR B3.

If a new source is added later, it becomes **V6.2**, not a modified V6.1.

The next notebook after a successful freeze should be the clean baseline training notebook that consumes this exact dataset tag.
